In [2]:
"""
data_preparation.py
-------------------
Módulo responsável pelo carregamento e preparação dos dados
para o diagnóstico de glaucoma a partir de retinografias.
Este script replica a metodologia descrita no artigo
"Diagnóstico de Glaucoma em Retinografias de Oftalmoscópio Portátil
Utilizando Ensemble Baseado em Transformers" (Costa et al., 2024).

Etapas:
1. Carregamento das imagens com torchvision.datasets.ImageFolder
2. Aplicação de transformações (Resize, Augmentation, Normalize)
3. Implementação manual de validação cruzada K-fold
4. Criação dos DataLoaders para treino e validação
"""

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


def get_transforms(model_name: str):
    """
    Retorna transformações SEM ToTensor() e SEM Normalize()
    O processor fará isso depois.
    """
    input_size = {
        "swinv2": 256,
        "beit": 224,
        "deit": 224,
        "vit": 224
    }.get(model_name.lower(), 224)
    
    transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        # NÃO adicionar ToTensor() aqui
        # O processor fará a conversão e normalização
    ])
    
    return transform


def load_dataset(data_root: str, model_name: str):
    """
    Carrega o dataset de imagens e aplica as transformações definidas.
    
    Parâmetros:
        data_root (str): caminho da pasta com subpastas /normal /glaucoma
        model_name (str): nome do modelo (para definir transformações)
    
    Retorna:
        dataset (torchvision.datasets.ImageFolder)
    """
    transform = get_transforms(model_name)
    dataset = datasets.ImageFolder(root=data_root, transform=transform)
    return dataset


def create_kfold_loaders(dataset, k_fold=5, batch_size=32, seed=42, collate_fn=None):
    """
    Implementa manualmente a validação cruzada K-fold,
    retornando DataLoaders de treino e validação para cada fold.
    
    Parâmetros:
        dataset: Dataset do torchvision
        k_fold (int): número de folds (padrão = 5)
        batch_size (int): tamanho do batch
        seed (int): semente aleatória para reprodutibilidade
        collate_fn (callable): função customizada para processar batches (opcional)
    
    Retorna:
        folds (list): lista de tuplas (train_loader, val_loader)
    """
    np.random.seed(seed)
    size = len(dataset)
    indices = np.arange(size)
    np.random.shuffle(indices)
    
    split_size = size // k_fold
    folds = []
    
    for fold in range(k_fold):
        val_start = fold * split_size
        val_end = val_start + split_size
        
        val_idx = indices[val_start:val_end]
        train_idx = np.concatenate((indices[:val_start], indices[val_end:]))
        
        train_subset = Subset(dataset, train_idx)
        val_subset = Subset(dataset, val_idx)
        
        # ✅ Adiciona collate_fn aos DataLoaders se fornecido
        train_loader = DataLoader(
            train_subset, 
            batch_size=batch_size, 
            shuffle=True,
            collate_fn=collate_fn
        )
        val_loader = DataLoader(
            val_subset, 
            batch_size=batch_size, 
            shuffle=False,
            collate_fn=collate_fn
        )
        
        folds.append((train_loader, val_loader))
        print(f"✅ Fold {fold+1}/{k_fold} criado — treino: {len(train_subset)} | validação: {len(val_subset)}")
    
    return folds


if __name__ == "__main__":
    # Exemplo de execução isolada
    dataset = load_dataset("data", model_name="vit")
    folds = create_kfold_loaders(dataset, k_fold=5, batch_size=32)

✅ Fold 1/5 criado — treino: 1600 | validação: 400
✅ Fold 2/5 criado — treino: 1600 | validação: 400
✅ Fold 3/5 criado — treino: 1600 | validação: 400
✅ Fold 4/5 criado — treino: 1600 | validação: 400
✅ Fold 5/5 criado — treino: 1600 | validação: 400


In [3]:
import torch
import torch.nn as nn
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification
)

MODEL_CONFIGS = {
    "vit": {
        "name": "google/vit-base-patch16-224",
        "input_size": 224
    },
    "swinv2": {
        "name": "microsoft/swinv2-base-patch4-window16-256",
        "input_size": 256
    },
    "deit": {
        "name": "facebook/deit-base-distilled-patch16-224",
        "input_size": 224
    },
    "beit": {
        "name": "microsoft/beit-base-patch16-224-pt22k-ft22k",
        "input_size": 224
    }
}


def load_transformer_model(model_name: str, num_classes: int = 2):
    """
    Carrega o modelo Transformer pré-treinado e adapta para classificação binária.

    Parâmetros:
        model_name (str): nome do modelo ('vit', 'swinv2', 'deit', 'beit')
        num_classes (int): número de classes (default = 2)

    Retorna:
        processor (AutoImageProcessor): pré-processador de imagem correspondente
        model (nn.Module): modelo ajustado para classificação binária
    """
    model_name = model_name.lower()
    assert model_name in MODEL_CONFIGS, f"Modelo '{model_name}' não suportado."

    cfg = MODEL_CONFIGS[model_name]
    print(f"🔹 Carregando modelo: {cfg['name']}")

    processor = AutoImageProcessor.from_pretrained(cfg["name"])
    model = AutoModelForImageClassification.from_pretrained(
        cfg["name"],
        num_labels=num_classes,
        ignore_mismatched_sizes=True  
    )

    if hasattr(model, "classifier"):
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
    elif hasattr(model, "head"):
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
    elif hasattr(model, "heads"):
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)

    return processor, model


def get_all_models(num_classes: int = 2):
    """
    Retorna todos os modelos usados no ensemble, já configurados.

    Retorna:
        models (dict): {'vit': model, 'swinv2': model, ...}
        processors (dict): {'vit': processor, ...}
    """
    models = {}
    processors = {}

    for name in MODEL_CONFIGS.keys():
        processor, model = load_transformer_model(name, num_classes=num_classes)
        models[name] = model
        processors[name] = processor

    return processors, models


if __name__ == "__main__":
    processors, models = get_all_models()
    for name, model in models.items():
        print(f"{name.upper()} carregado com sucesso. Parâmetros: {sum(p.numel() for p in model.parameters()):,}")


/media/henrique/Projetos/GlaucoVision/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔹 Carregando modelo: google/vit-base-patch16-224


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔹 Carregando modelo: microsoft/swinv2-base-patch4-window16-256


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of Swinv2ForImageClassification were not initialized from the model checkpoint at microsoft/swinv2-base-patch4-window16-256 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔹 Carregando modelo: facebook/deit-base-distilled-patch16-224


Some weights of DeiTForImageClassificationWithTeacher were not initialized from the model checkpoint at facebook/deit-base-distilled-patch16-224 and are newly initialized because the shapes did not match:
- cls_classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- cls_classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- distillation_classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- distillation_classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔹 Carregando modelo: microsoft/beit-base-patch16-224-pt22k-ft22k


Some weights of BeitForImageClassification were not initialized from the model checkpoint at microsoft/beit-base-patch16-224-pt22k-ft22k and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([21841, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([21841]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


VIT carregado com sucesso. Parâmetros: 85,800,194
SWINV2 carregado com sucesso. Parâmetros: 86,895,866
DEIT carregado com sucesso. Parâmetros: 85,803,268
BEIT carregado com sucesso. Parâmetros: 85,763,522


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from tqdm import tqdm
import os

# from data_preparation import load_dataset, create_kfold_loaders
# from model import load_transformer_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def create_collate_fn(processor):
    """
    Cria uma função collate customizada que processa imagens PIL com o processor.
    """
    def collate_fn(batch):
        images = [item[0] for item in batch]
        labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
        
        # Processa as imagens com o processor
        processed = processor(images=images, return_tensors="pt")
        pixel_values = processed['pixel_values']
        
        return pixel_values, labels
    
    return collate_fn


def train_one_fold(model, processor, train_loader, val_loader, optimizer, criterion, model_name, fold_idx, epochs=10, save_dir="results"):
    """
    Executa o treinamento e validação para um único fold e salva as probabilidades.
    
    Parâmetros:
        model: modelo Transformer
        processor: AutoImageProcessor do modelo (para normalização correta)
        train_loader: DataLoader de treino
        val_loader: DataLoader de validação
        optimizer: otimizador
        criterion: função de perda
        model_name: nome do modelo
        fold_idx: índice do fold atual
        epochs: número de épocas
        save_dir: diretório para salvar resultados
    """
    model.to(DEVICE)
    history = {"train_loss": [], "val_loss": []}
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for pixel_values, labels in train_loader:
            pixel_values = pixel_values.to(DEVICE)
            labels = labels.to(DEVICE)
                    
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        history["train_loss"].append(train_loss)

        # Validação
        model.eval()
        val_loss = 0.0
        preds, probs, gts = [], [], []
        
        with torch.no_grad():
            for pixel_values, labels in val_loader:
                pixel_values = pixel_values.to(DEVICE)
                labels = labels.to(DEVICE)
                
                outputs = model(pixel_values=pixel_values).logits
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                pred = torch.argmax(outputs, dim=1)
                preds.extend(pred.cpu().numpy())
                probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())
                gts.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        history["val_loss"].append(val_loss)

        acc = accuracy_score(gts, preds)
        prec = precision_score(gts, preds, zero_division=0)
        rec = recall_score(gts, preds, zero_division=0)
        f1 = f1_score(gts, preds, zero_division=0)
        auc = roc_auc_score(gts, probs)

        print(f"📘 Época {epoch+1}/{epochs} | Loss treino: {train_loss:.4f} | Loss val: {val_loss:.4f}")
        print(f"➡️  ACC={acc*100:.2f}% | PREC={prec*100:.2f}% | REC={rec*100:.2f}% | F1={f1*100:.2f}% | AUC={auc*100:.2f}%")

    np.save(f"{save_dir}/probs_{model_name}_fold{fold_idx}.npy", np.array(probs))
    np.save(f"{save_dir}/gts_fold{fold_idx}.npy", np.array(gts))

    return model, (acc, prec, rec, f1, auc)


def kfold_training(data_root="data", model_name="vit", k_fold=5, epochs=10, lr=1e-4, batch_size=32, save_dir="results"):
    """
    Executa o treinamento com validação cruzada K-fold para um modelo específico
    e salva os resultados em um arquivo TXT.
    """
    print(f"\n🚀 Iniciando K-Fold Training ({k_fold} folds) — Modelo: {model_name.upper()}\n")
    
    # Carrega o processor ANTES de criar os DataLoaders
    processor, _ = load_transformer_model(model_name, num_classes=2)
    
    dataset = load_dataset(data_root, model_name)
    folds = create_kfold_loaders(
        dataset, 
        k_fold=k_fold, 
        batch_size=batch_size,
        collate_fn=create_collate_fn(processor)  # ✅ Adiciona collate_fn customizado
    )

    os.makedirs(save_dir, exist_ok=True)
    txt_file_path = os.path.join(save_dir, f"results_{model_name}.txt")

    all_metrics = []

    with open(txt_file_path, "w", encoding='utf-8') as f:
        f.write(f"📊 Resultados K-Fold ({k_fold} folds) — Modelo: {model_name.upper()}\n\n")

        for fold_idx, (train_loader, val_loader) in enumerate(folds, start=1):
            f.write(f"==============================\n")
            f.write(f"🔹 FOLD {fold_idx}/{k_fold}\n")
            f.write(f"==============================\n")

            # Recarrega o modelo para cada fold
            _, model = load_transformer_model(model_name, num_classes=2)
            model.to(DEVICE)

            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

            trained_model, metrics = train_one_fold(
                model, processor, train_loader, val_loader, optimizer, criterion,
                model_name, fold_idx, epochs, save_dir
            )
            all_metrics.append(metrics)

            acc, prec, rec, f1, auc = metrics
            f.write(f"Accuracy:     {acc*100:.2f}%\n")
            f.write(f"Precision:    {prec*100:.2f}%\n")
            f.write(f"Recall:       {rec*100:.2f}%\n")
            f.write(f"F1-Score:     {f1*100:.2f}%\n")
            f.write(f"AUC:          {auc*100:.2f}%\n\n")

        all_metrics = np.array(all_metrics)
        mean_metrics = all_metrics.mean(axis=0)
        std_metrics = all_metrics.std(axis=0)

        f.write(f"✅ MÉDIA FINAL ({model_name.upper()} — {k_fold} folds):\n")
        f.write(f"Accuracy:     {mean_metrics[0]*100:.2f}% ± {std_metrics[0]*100:.2f}%\n")
        f.write(f"Precision:    {mean_metrics[1]*100:.2f}% ± {std_metrics[1]*100:.2f}%\n")
        f.write(f"Recall:       {mean_metrics[2]*100:.2f}% ± {std_metrics[2]*100:.2f}%\n")
        f.write(f"F1-Score:     {mean_metrics[3]*100:.2f}% ± {std_metrics[3]*100:.2f}%\n")
        f.write(f"AUC:          {mean_metrics[4]*100:.2f}% ± {std_metrics[4]*100:.2f}%\n")

    print(f"\n✅ Resultados salvos em {txt_file_path}")
    return all_metrics


def full_experiment(data_root="data", k_fold=5, epochs=10, lr=1e-4, batch_size=32):
    """
    Executa o experimento completo com todos os modelos Transformers.
    (VIT, DEIT, BEIT, SWINV2) e salva resultados em TXT individual.
    """
    # models = ["vit", "deit", "beit", "swinv2"]
    models = ["swinv2"]
    all_results = {}

    for model_name in models:
        print("\n" + "=" * 70)
        print(f"🚀 Iniciando treinamento completo para {model_name.upper()}")
        print("=" * 70)
        results = kfold_training(
            data_root=data_root,
            model_name=model_name,
            k_fold=k_fold,
            epochs=epochs,
            lr=lr,
            batch_size=batch_size,
            save_dir="results"
        )
        all_results[model_name] = results

    print("\n🎯 Treinamento completo finalizado para todos os modelos!")
    return all_results


if __name__ == "__main__":
    full_experiment(
        data_root="data",
        k_fold=5,
        epochs=10,
        lr=1e-4,
        batch_size=16
    )


🚀 Iniciando treinamento completo para SWINV2

🚀 Iniciando K-Fold Training (5 folds) — Modelo: SWINV2

🔹 Carregando modelo: microsoft/swinv2-base-patch4-window16-256


Some weights of Swinv2ForImageClassification were not initialized from the model checkpoint at microsoft/swinv2-base-patch4-window16-256 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Fold 1/5 criado — treino: 1600 | validação: 400
✅ Fold 2/5 criado — treino: 1600 | validação: 400
✅ Fold 3/5 criado — treino: 1600 | validação: 400
✅ Fold 4/5 criado — treino: 1600 | validação: 400
✅ Fold 5/5 criado — treino: 1600 | validação: 400
🔹 Carregando modelo: microsoft/swinv2-base-patch4-window16-256


Some weights of Swinv2ForImageClassification were not initialized from the model checkpoint at microsoft/swinv2-base-patch4-window16-256 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import os

def ensemble_predictions(prob_files_dict, gts_file_dict, save_dir="results"):
    """
    Calcula o Ensemble por média das probabilidades entre múltiplos modelos
    e salva os resultados em um arquivo TXT.

    Parâmetros:
        prob_files_dict (dict): {'vit': ['vit_fold1.npy', 'vit_fold2.npy', ...], ...}
        gts_file_dict (dict): {'vit': ['gts_fold1.npy', ...]} (iguais entre modelos)
        save_dir (str): pasta para salvar o arquivo TXT

    Retorna:
        results (np.array): métricas por fold [acc, prec, rec, f1, auc]
    """
    os.makedirs(save_dir, exist_ok=True)
    txt_file_path = os.path.join(save_dir, "ensemble_results.txt")

    num_folds = len(next(iter(prob_files_dict.values())))
    results = []

    print(f"\n🚀 Iniciando Ensemble com {len(prob_files_dict)} modelos e {num_folds} folds...\n")

    with open(txt_file_path, "w") as f:
        f.write(f"📊 Resultados Ensemble ({len(prob_files_dict)} modelos, {num_folds} folds)\n\n")

        for fold in range(num_folds):
            f.write(f"==============================\n")
            f.write(f"🔹 Fold {fold + 1}/{num_folds}\n")
            f.write(f"==============================\n")

            probs_models = []
            for model_name, files in prob_files_dict.items():
                probs = np.load(files[fold])
                probs_models.append(probs)
                f.write(f"{model_name.upper()} carregado (shape: {probs.shape})\n")
                print(f"  → {model_name.upper()} carregado para o fold {fold + 1} (shape: {probs.shape})")

            probs_mean = np.mean(probs_models, axis=0)
            preds_ensemble = (probs_mean >= 0.5).astype(int)
            gts = np.load(gts_file_dict[list(gts_file_dict.keys())[0]][fold])

            acc = accuracy_score(gts, preds_ensemble)
            prec = precision_score(gts, preds_ensemble)
            rec = recall_score(gts, preds_ensemble)
            f1 = f1_score(gts, preds_ensemble)
            auc = roc_auc_score(gts, probs_mean)

            f.write(f"Accuracy:     {acc*100:.2f}%\n")
            f.write(f"Precision:    {prec*100:.2f}%\n")
            f.write(f"Recall:       {rec*100:.2f}%\n")
            f.write(f"F1-Score:     {f1*100:.2f}%\n")
            f.write(f"AUC:          {auc*100:.2f}%\n\n")

            print(f"  ACC={acc*100:.2f}% | PREC={prec*100:.2f}% | REC={rec*100:.2f}% | F1={f1*100:.2f}% | AUC={auc*100:.2f}%")
            results.append([acc, prec, rec, f1, auc])

        results = np.array(results)
        mean_metrics = results.mean(axis=0)
        std_metrics = results.std(axis=0)

        f.write("📊 MÉTRICAS MÉDIAS DO ENSEMBLE:\n")
        f.write(f"Accuracy:     {mean_metrics[0]*100:.2f}% ± {std_metrics[0]*100:.2f}%\n")
        f.write(f"Precision:    {mean_metrics[1]*100:.2f}% ± {std_metrics[1]*100:.2f}%\n")
        f.write(f"Recall:       {mean_metrics[2]*100:.2f}% ± {std_metrics[2]*100:.2f}%\n")
        f.write(f"F1-Score:     {mean_metrics[3]*100:.2f}% ± {std_metrics[3]*100:.2f}%\n")
        f.write(f"AUC:          {mean_metrics[4]*100:.2f}% ± {std_metrics[4]*100:.2f}%\n")

    print(f"\n✅ Resultados do ensemble salvos em {txt_file_path}")
    return results


if __name__ == "__main__":
    prob_files = {
        "vit": [f"results/probs_vit_fold{i}.npy" for i in range(1, 6)],
        "swinv2": [f"results/probs_swinv2_fold{i}.npy" for i in range(1, 6)],
        "deit": [f"results/probs_deit_fold{i}.npy" for i in range(1, 6)],
        "beit": [f"results/probs_beit_fold{i}.npy" for i in range(1, 6)],
    }

    gts_files = {
        "vit": [f"results/gts_fold{i}.npy" for i in range(1, 6)]
    }

    ensemble_predictions(prob_files, gts_files)
